# 9 - Statistical Analysis

## Overview

Statistical significance testing: paired t-test, Wilcoxon signed-rank test, McNemar's test.

Compare XGBoost vs LightGBM, and FL models vs baselines.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import SEED

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.metrics import f1_score, accuracy_score

import xgboost as xgb
import lightgbm as lgb

# Paths
PREPROCESSED_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed"
LOGS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/logs"
FIGURES_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/figures"

print("Statistical analysis notebook initialized.")

## 1. Load Data & Train Models

In [ ]:
# Load data
print("Loading data...")
train_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"))
test_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "test.parquet"))

# Load metadata
with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
num_classes = metadata['num_classes']

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test = test_df[feature_cols].values
y_test = test_df['label'].values

# Load label encoder for class names
import joblib
le = joblib.load(os.path.join(PREPROCESSED_DIR, "label_encoder.pkl"))

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Train models
from sklearn.utils import resample
from src.config import XGB_PARAMS, LGB_PARAMS
sample_size = min(200_000, len(X_train))
X_tr, y_tr = resample(X_train, y_train, n_samples=sample_size, random_state=42)
print(f"Training on {sample_size:,} samples (was {len(X_train):,})")

xgb_params = XGB_PARAMS.copy()
xgb_params['n_jobs'] = 8
xgb_params.pop('n_estimators', None)

print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(**xgb_params, n_estimators=100)
xgb_model.fit(X_tr, y_tr, verbose=False)
xgb_pred = xgb_model.predict(X_test)
print(f"XGBoost trained. Test F1: {f1_score(y_test, xgb_pred, average='macro'):.4f}")

lgb_params = LGB_PARAMS.copy()
lgb_params['n_jobs'] = 8
lgb_params.pop('n_estimators', None)

print("\nTraining LightGBM...")
lgb_model = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
lgb_model.fit(X_tr, y_tr)
lgb_pred = lgb_model.predict(X_test)
print(f"LightGBM trained. Test F1: {f1_score(y_test, lgb_pred, average='macro'):.4f}")

## 2. Per-Sample Predictions for Statistical Tests

In [ ]:
# Get per-sample F1 for each model
# We'll use bootstrap samples for statistical testing

def compute_bootstrap_scores(X, y, model, n_bootstrap=30, sample_size=10000, seed=42):
    """Compute F1 scores on bootstrap samples."""
    np.random.seed(seed)
    scores = []
    
    for i in range(n_bootstrap):
        idx = np.random.choice(len(X), size=sample_size, replace=True)
        X_boot, y_boot = X[idx], y[idx]
        
        y_pred = model.predict(X_boot)
        f1 = f1_score(y_boot, y_pred, average='macro')
        scores.append(f1)
    
    return np.array(scores)

print("Computing bootstrap F1 scores...")

# Bootstrap XGBoost
print("  XGBoost bootstrap...")
xgb_bootstrap = compute_bootstrap_scores(X_test, y_test, xgb_model, n_bootstrap=30, sample_size=10000, seed=SEED)

# Bootstrap LightGBM
print("  LightGBM bootstrap...")
lgb_bootstrap = compute_bootstrap_scores(X_test, y_test, lgb_model, n_bootstrap=30, sample_size=10000, seed=SEED)

print(f"\nBootstrap F1 scores computed: {len(xgb_bootstrap)} samples each")

## 3. Paired T-Test

In [ ]:
# Paired t-test
print("\n" + "=" * 60)
print("PAIRED T-TEST")
print("=" * 60)

t_stat, t_p_value = stats.ttest_rel(xgb_bootstrap, lgb_bootstrap)

print(f"\nXGBoost mean F1: {xgb_bootstrap.mean():.4f} ± {xgb_bootstrap.std():.4f}")
print(f"LightGBM mean F1: {lgb_bootstrap.mean():.4f} ± {lgb_bootstrap.std():.4f}")
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {t_p_value:.6f}")

alpha = 0.05
if t_p_value < alpha:
    print(f"\nResult: SIGNIFICANT difference (p < {alpha})")
    if xgb_bootstrap.mean() > lgb_bootstrap.mean():
        print("Winner: XGBoost")
    else:
        print("Winner: LightGBM")
else:
    print(f"\nResult: NO significant difference (p >= {alpha})")

## 4. Wilcoxon Signed-Rank Test

In [ ]:
# Wilcoxon signed-rank test (non-parametric)
print("\n" + "=" * 60)
print("WILCOXON SIGNED-RANK TEST")
print("=" * 60)

w_stat, w_p_value = stats.wilcoxon(xgb_bootstrap, lgb_bootstrap)

print(f"\nStatistic: {w_stat:.4f}")
print(f"P-value: {w_p_value:.6f}")

if w_p_value < alpha:
    print(f"\nResult: SIGNIFICANT difference (p < {alpha})")
else:
    print(f"\nResult: NO significant difference (p >= {alpha})")

## 5. McNemar's Test

In [ ]:
# McNemar's test on predictions
print("\n" + "=" * 60)
print("MCNEMAR'S TEST")
print("=" * 60)

# Create contingency table
# Count samples where models disagree
xgb_correct = (xgb_pred == y_test)
lgb_correct = (lgb_pred == y_test)

n00 = np.sum(~xgb_correct & ~lgb_correct)  # Both wrong
n01 = np.sum(~xgb_correct & lgb_correct)   # XGB wrong, LGB correct
n10 = np.sum(xgb_correct & ~lgb_correct)  # XGB correct, LGB wrong
n11 = np.sum(xgb_correct & lgb_correct)   # Both correct

print(f"\nContingency Table:")
print(f"                    LGB Correct   LGB Wrong")
print(f"  XGB Correct     {n11:10d}     {n10:10d}")
print(f"  XGB Wrong       {n01:10d}     {n00:10d}")

# McNemar's test (proper: only discordant pairs)
# H0: both models have equal error rates
n_discordant = n01 + n10
if n_discordant < 25:
    from scipy.stats import binom_test
    m_p_value = binom_test(n01, n_discordant, p=0.5)
else:
    from scipy.stats import chi2
    chi2_stat = (abs(n01 - n10) - 1)**2 / n_discordant
    m_p_value = 1 - chi2.cdf(chi2_stat, df=1)

print(f"\nDiscordant pairs (n01 + n10): {n_discordant}")
print(f"P-value: {m_p_value:.6f}")

if m_p_value < alpha:
    print(f"\nResult: SIGNIFICANT difference in prediction patterns (p < {alpha})")
    if n10 > n01:
        print("XGBoost makes more correct predictions than LightGBM")
    else:
        print("LightGBM makes more correct predictions than XGBoost")
else:
    print(f"\nResult: NO significant difference in prediction patterns (p >= {alpha})")

## 6. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap confidence intervals
print("\n" + "=" * 60)
print("BOOTSTRAP CONFIDENCE INTERVALS (95%)")
print("=" * 60)

confidence = 0.95

def compute_ci(scores, confidence=0.95):
    """Compute confidence interval."""
    alpha = 1 - confidence
    lower = np.percentile(scores, alpha/2 * 100)
    upper = np.percentile(scores, (1 - alpha/2) * 100)
    return lower, upper

# XGBoost CI
xgb_lower, xgb_upper = compute_ci(xgb_bootstrap, confidence)
print(f"\nXGBoost F1: {xgb_bootstrap.mean():.4f} ({confidence*100:.0f}% CI: [{xgb_lower:.4f}, {xgb_upper:.4f}])")

# LightGBM CI
lgb_lower, lgb_upper = compute_ci(lgb_bootstrap, confidence)
print(f"LightGBM F1: {lgb_bootstrap.mean():.4f} ({confidence*100:.0f}% CI: [{lgb_lower:.4f}, {lgb_upper:.4f}])")

# Difference CI
diff_bootstrap = xgb_bootstrap - lgb_bootstrap
diff_lower, diff_upper = compute_ci(diff_bootstrap, confidence)
print(f"\nDifference (XGB - LGB): {diff_bootstrap.mean():.4f} (95% CI: [{diff_lower:.4f}, {diff_upper:.4f}])")

## 7. Visualization

In [ ]:
# Plot bootstrap distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XGBoost distribution
ax = axes[0]
ax.hist(xgb_bootstrap, bins=15, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(xgb_bootstrap.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {xgb_bootstrap.mean():.4f}')
ax.axvline(xgb_lower, color='orange', linestyle=':', linewidth=1.5, label=f'95% CI')
ax.axvline(xgb_upper, color='orange', linestyle=':', linewidth=1.5)
ax.set_xlabel('F1 Macro')
ax.set_ylabel('Frequency')
ax.set_title('XGBoost Bootstrap F1 Distribution')
ax.legend()

# LightGBM distribution
ax = axes[1]
ax.hist(lgb_bootstrap, bins=15, alpha=0.7, color='coral', edgecolor='black')
ax.axvline(lgb_bootstrap.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {lgb_bootstrap.mean():.4f}')
ax.axvline(lgb_lower, color='orange', linestyle=':', linewidth=1.5, label=f'95% CI')
ax.axvline(lgb_upper, color='orange', linestyle=':', linewidth=1.5)
ax.set_xlabel('F1 Macro')
ax.set_ylabel('Frequency')
ax.set_title('LightGBM Bootstrap F1 Distribution')
ax.legend()

# Difference distribution
ax = axes[2]
ax.hist(diff_bootstrap, bins=15, alpha=0.7, color='green', edgecolor='black')
ax.axvline(diff_bootstrap.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {diff_bootstrap.mean():.4f}')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('F1 Difference (XGB - LGB)')
ax.set_ylabel('Frequency')
ax.set_title('Difference Distribution')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '09_statistical_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved: results/figures/09_statistical_analysis.png")

## 8. Save Results

In [ ]:
# Save statistical analysis results
stat_results = {
    'xgb_bootstrap': {
        'mean': float(xgb_bootstrap.mean()),
        'std': float(xgb_bootstrap.std()),
        'ci_lower': float(xgb_lower),
        'ci_upper': float(xgb_upper)
    },
    'lgb_bootstrap': {
        'mean': float(lgb_bootstrap.mean()),
        'std': float(lgb_bootstrap.std()),
        'ci_lower': float(lgb_lower),
        'ci_upper': float(lgb_upper)
    },
    'paired_ttest': {
        't_statistic': float(t_stat),
        'p_value': float(t_p_value),
        'significant': bool(t_p_value < 0.05)
    },
    'wilcoxon': {
        'statistic': float(w_stat),
        'p_value': float(w_p_value),
        'significant': bool(w_p_value < 0.05)
    },
    'mcnemar': {
        'n01': int(n01),
        'n10': int(n10),
        'discordant_pairs': int(n01 + n10),
        'p_value': float(m_p_value),
        'significant': bool(m_p_value < 0.05)
    },
    'difference': {
        'mean': float(diff_bootstrap.mean()),
        'ci_lower': float(diff_lower),
        'ci_upper': float(diff_upper)
    }
}

with open(os.path.join(LOGS_DIR, '09_statistical_analysis.json'), 'w') as f:
    json.dump(stat_results, f, indent=2)

print("Statistical analysis saved: results/logs/09_statistical_analysis.json")

In [ ]:
print("## Summary")
print()
print("### Statistical Tests Results:")
print()
print(f"| Test | Statistic | P-value | Significant? |")
print(f"|------|-----------|---------|---------------|")
print(f"| Paired T-Test | {t_stat:.4f} | {t_p_value:.6f} | {'Yes' if t_p_value < 0.05 else 'No'} |")
print(f"| Wilcoxon | {w_stat:.4f} | {w_p_value:.6f} | {'Yes' if w_p_value < 0.05 else 'No'} |")
print(f"| McNemar | {n01 + n10} discordant | {m_p_value:.6f} | {'Yes' if m_p_value < 0.05 else 'No'} |")
print()
print("### Bootstrap 95% CI:")
print(f"- XGBoost: [{xgb_lower:.4f}, {xgb_upper:.4f}]")
print(f"- LightGBM: [{lgb_lower:.4f}, {lgb_upper:.4f}]")
print()
print("### Conclusion:")
xgb_mean = xgb_bootstrap.mean()
lgb_mean = lgb_bootstrap.mean()
diff = xgb_mean - lgb_mean
if t_p_value < 0.05:
    winner = "XGBoost" if diff > 0 else "LightGBM"
    print(f"{winner} significantly outperforms the other model (p = {t_p_value:.6f}, "
          f"difference = {abs(diff):.4f} F1).")
else:
    print("Both models perform similarly with no statistically significant difference "
          f"(p = {t_p_value:.4f}).")
print()
print("**Pipeline Complete!**")